# Simulation types (advanced)
The ability prepare, propagate, and measure quantum states in simulations is a core capability of pyGSTi.  We call the calculation of circuit outcome probabilities *forward simulation*, and this type of calculation lies at the heart of gate set tomography and other characterization protocols which need to compare a model's predictions with actual data.  As you're probably already encountered through other tutorials, `Model` objects have a `.probabilities(circuit)` method that performs forward simulation (computes the outcome probabilities of `circuit`).  

What is not so apparent is that there can be several different types of computational "engines" under the hood of a `Model` that do the heavy lifting within a call to `probs`.  **Understanding the different types of forward simulation engines in pyGSTi is the topic of this tutorial.**

First, let's lay down a bit of background.  A `Model` contains a `.evotype` attribute that describes what type of underlying *state representation* is being stored and propagated by the model - the *evolution type*.  Allowed values are:
- `"densitymx"`: a *mixed* state is propagated as a vectorized density matrix (of length $4^n$ for $n$ qubits) in a Hermitian basis (so its elements are *real*).  If an operation is represented as a dense (real) matrix, its shape is $4^n \times 4^n$.  This is the default evolution type in pyGSTi.
- `"statevec"`: a *pure* state is propagated as a complex state vector (of length $2^n$ for $n$ qubits).  Dense operations representations are $2^n \times 2^n$ unitary matrices in this case.  Because of the lower dimensionality, state-vector propagation is faster than density-matrix propagation, and is therefore preferable when all preparations and POVM effects are pure states and projections, and all operations are unitary.
- `"stabilizer"`: a *stabilizer* state is propagated by keeping track of its stabilizer (and anti-stabilizer) group generators.  This only requires memory that scales with $n$ for $n$ qubits, and so this state representation offers significantly more efficient simulation than the two aforementioned evolution types.  The caveat is preparations must prepare a stabilizer state, POVMs must measure in the computational $z$-basis, and *all operations must be Clifford elements*, mapping Pauli group elements to Pauli group elements.
- `"[densitymx|statevec|stabilizer]_slow"`: a pure-Python "slow" version of each of the above evolution types exists so that building pyGSTi's Cython extensions is optional.  If you're unable to build the Cython extensions for any reason, any of these `_slow` evolution types can still be used.  Despite the name, the relative speed of the `_slow` and Cython-extension versions depends on workload and hardware: the `_slow` (numpy-based) implementation can sometimes outperform the Cython one, particularly for circuits with many idle qubits (see [pyGSTi issue #713](https://github.com/sandialabs/pyGSTi/issues/713)).
- `"default"`: whatever value is in `pygsti.evotypes.Evotype.default_evotype`.  When pygsti is imported, this is set to `densitymx` if the Cython extensions are built and `densitymx_slow` otherwise.  Users may modify this value to set a different default, which can be convenient since many objects that require an evolution type default to `"default"`.

Every operator (state preparation, POVM, gate/layer operation) within a `Model` also has an evolution type, and it must match the parent `Model`'s `.evotype`.  (You usually don't need to worry about this; `Model` objects are created with a given `evotype` and their contained elements are created to match that type.)  

Related to their `evotype`, `Model` objects also have a `sim` attribute.  This attribute is the forward simulation engine that is used when the model is asked to compute circuit probabilities.  It is an instance of a subclass of `pygsti.forwardsims.ForwardSimulator`.  There are several different types of forward simulators that the `sim` attribute can be set to, but each type is only compatible with certain evolution types (`evotype` value):
- `MapForwardSimulator` or `"map"`: propagate a `"densitymx"`, `"statevec"`, `"stabilizer"` state by repeatedly acting with circuit layer operators, treated as maps from an input to an output state.  When there is a dense layer-operation matrix for each circuit layer, this engine repeatedly performs matrix-vector products (one per layer, between the operation matrix and column-vector state) and finally contracts the result with each row-vector POVM effect to get outcome probabilities.  This is the most straightforward and intuitive of the forward simulation engines, so much so that you may be thinking "what else would you do?".  Keep reading :)
- `MatrixForwardSimulator` or `"matrix"`: propagate a `"densitymx"` or `"statevec"` state by composing a dense operation matrix for the entire circuit and then applying that "circuit map" to the input state.  Each circuit layer *is represented as a dense matrix* when using this forward simulation type, as it multiplies together the matrices of each layer (in reverse order!) to get a single "circuit matrix* and then contracts this matrix between the (column vector) state preparation and each of the (row vector) POVM effects (and norm-squaring the result when the evolution type is `"statevec"`) to get outcome probabilities.  At first glance this seems like a very inefficient way to compute probabilities since it performs matrix-matrix instead of matrix-vector multiplications, and it is true that this method should not be used with many-qubit models.  However, for lower dimensional Hilbert spaces (1-2 qubits in practice) the ability to cache and reuse intermediate results can make this forward simulator faster than the `"map"` type when the outcome probabilities of *many* circuits are needed at once.
- `TermForwardSimulator`: computes circuit outcome probabilites by evaluating a truncated path integral assembled by Taylor-expanding each circuit operation and keeping terms that are below some Taylor-term order or that have a weight below some threshold.  Each path is evaluated by propagating a pure state under unitary actions, and so this forward simulator works with `"statevec"` and `"stabilizer"` evotypes.
- `CHPForwardSimulator` or `"chp"`: simulates Clifford circuits using Scott Aaronson's CHP program.

Thus, by setting a model's `evotype` and `sim` you can specify how circuit-probability-computation is implemented.  But when and how do you set these values?  The `evotype` of a `Model` is almost always set for good when the object is created.  The simulator *can* be changed assigning a new forward simulator object to the `sim` property of a `Model`, but it's usually set to an appropriate value at object-creation time and doesn't need to be altered.  In most model construction functions (in `pygsti.models.modelconstruction`) there are `evotype` and `simulator` arguments that determine these values.  

Often the `simulator` argument can be left as `"auto"`, which currently selects the `"map"` simulator.  The `"matrix"` simulator can still be useful in small (1–2 qubit) Hilbert spaces where dense process-matrix caching pays off across very large circuit batches, but it is no longer the automatic choice.  The default for an `evotype` argument is usually the string literal `"default"`, which ends up getting mapped to a string appropriate for the parameterization of the noise model (`"densitymx"` for open-system Markovian dynamics).
 
 
Below, we'll demonstrate the use of different evolution and forward-simulation types using a local-noise model on 5 and then 10 qubits.  

## 5 qubits
First, let's generate a random circuit using the `pygsti.algorithms.randomcicuit` module.  Random circuits are primarily used in randomized benchmarking (see the [RB tutorial](../../guides/rb/HowRBWorks) for more information about running RB); here we just use it as an easy way to create an example circuit without having to write down one by hand.

In [1]:
import pygsti, time
import numpy as np

from pygsti.processors import QubitProcessorSpec

n_qubits = 5
ps = QubitProcessorSpec(num_qubits=n_qubits, gate_names=['Gx','Gy','Gcnot'],
                        availability={'Gcnot': [(i,i+1) for i in range(n_qubits-1)]})

c = pygsti.algorithms.randomcircuit.create_random_circuit(ps, length=20)
print(c)

Qubit 0 ---|Gx|-|C1|-|Gy|-|Gx|-|Gx|-|Gy|-|Gy|-|C1|-|Gy|-|C1|-|C1|-|Gy|-|Gy|-|Gx|-|Gy|-|C1|-|C1|-|Gy|-|C1|-|Gy|---
Qubit 1 ---|Gx|-|T0|-|Gy|-|C2|-|Gy|-|Gy|-|Gy|-|T0|-|Gy|-|T0|-|T0|-|C2|-|Gx|-|Gy|-|Gx|-|T0|-|T0|-|Gy|-|T0|-|C2|---
Qubit 2 ---|Gy|-|Gx|-|C3|-|T1|-|Gy|-|Gx|-|Gy|-|Gx|-|C3|-|C3|-|Gy|-|T1|-|Gx|-|Gx|-|Gy|-|C3|-|C3|-|Gx|-|Gx|-|T1|---
Qubit 3 ---|C4|-|C4|-|T2|-|C4|-|C4|-|C4|-|Gx|-|C4|-|T2|-|T2|-|Gx|-|Gx|-|Gy|-|C4|-|Gy|-|T2|-|T2|-|Gy|-|C4|-|Gx|---
Qubit 4 ---|T3|-|T3|-|Gx|-|T3|-|T3|-|T3|-|Gx|-|T3|-|Gx|-|Gy|-|Gy|-|Gy|-|Gx|-|T3|-|Gx|-|Gy|-|Gy|-|Gx|-|T3|-|Gx|---



Propagate a **density matrix** (`"densitymx"` evolution type) using the **matrix-matrix multiplying** forward simulator (`"matrix"`):

In [2]:
mdl = pygsti.models.create_crosstalk_free_model(ps, simulator="matrix")
print("Gx gate is a ",type(mdl.operation_blks['gates']['Gx']))
t0 = time.time()
out = mdl.probabilities(c)
print("%d probabilities computed in %.3fs" % (len(out), time.time()-t0))

Gx gate is a  <class 'pygsti.modelmembers.operations.staticunitaryop.StaticUnitaryOp'>
32 probabilities computed in 10.049s


We can also propagate a density matrix using the **matrix-vector multiplying** forward simulator (`"map"`).  This forward simulator is much faster than `"matrix"` for even several (5) qubits.  This is why `"map"` is the simulator pyGSTi selects when the `simulator` argument is left as `"auto"`.

In [3]:
mdl = pygsti.models.create_crosstalk_free_model(ps, simulator="map")
t0 = time.time()
out2 = mdl.probabilities(c)
print("%d probabilities computed in %.3fs" % (len(out2), time.time()-t0))
assert(all([np.isclose(out[k],out2[k]) for k in out])) # check that the probabilites are the same

32 probabilities computed in 0.009s


If we don't need to consider mixed states, we can represent the quantum state using a **state vector** (`"statevec"`, selected by the `"static unitary"` parameterization type) and use either the matrix-matrix or matrix-vector product simulation types (the latter is again considerably faster even for just 5 qubits):

In [4]:
# TODO: Unitary evolution is not yet supported for matrix
#mdl = pygsti.models.create_crosstalk_free_model(ps, ideal_gate_type='static unitary',
#                                                evotype='statevec', simulator='matrix')
#t0 = time.time()
#out3 = mdl.probabilities(c)
#print("Mat-mat: %d probabilities in %.3fs" % (len(out3), time.time()-t0))
#assert(all([np.isclose(out[k],out3[k]) for k in out])) # check that the probabilites are the same

mdl = pygsti.models.create_crosstalk_free_model(ps, ideal_gate_type='static unitary',
                                                evotype='statevec', simulator='map')
t0 = time.time()
out4 = mdl.probabilities(c)
print("Mat-vec: %d probabilities in %.3fs" % (len(out4), time.time()-t0))
assert(all([np.isclose(out[k],out4[k]) for k in out])) # check that the probabilites are the same

Mat-vec: 32 probabilities in 0.002s


Finally, if all the gates are **Clifford** operations (as they are in this case), we can use the `"clifford"` parameterization to propagate a `"stabilizer"` state.  Only the `"map"` simulation type is compatible with the `"stabilizer"` evolution type (selected automatically).

In [5]:
mdl = pygsti.models.create_crosstalk_free_model(ps, ideal_gate_type='static clifford',
                                                simulator='map', evotype='stabilizer')
print("Gx gate is a ",type(mdl.operation_blks['gates']['Gx']))
t0 = time.time()
out5 = mdl.probabilities(c)
print("%d probabilities in %.3fs" % (len(out5), time.time()-t0))
assert(all([np.isclose(out[k],out5[k]) for k in out])) # check that the probabilites are the same

Gx gate is a  <class 'pygsti.modelmembers.operations.staticcliffordop.StaticCliffordOp'>
32 probabilities in 0.002s


## 10 qubits
Let's create a function to compare the above methods for a given number of qubits.  We'll automatically exclude the `"densitymx"`-`"matrix"` case when the number of qubits is greater than 5 as we know this is getting slow at this point.  At 10 qubits, the stabilizer and state-vector simulations are of comparable runtime (though this is largely due to the fact that *all* the outcomes are always computed - see below).

In [6]:
import pygsti, time

def compare_calc_methods(n_qubits):
    print("---- Comparing times for %d qubits (%d outcomes) ----" % (n_qubits,2**n_qubits))
    t0=time.time()
    ps = QubitProcessorSpec(n_qubits, gate_names=['Gx','Gy','Gcnot'],
                                  availability={'Gcnot': [(i,i+1) for i in range(n_qubits-1)]})
    print("Create processor spec: %.3fs" % (time.time()-t0))

    c = pygsti.algorithms.randomcircuit.create_random_circuit(ps, 20)
    print("Random Circuit:")
    print(c)

    if n_qubits <= 5:
        mdl = pygsti.models.create_crosstalk_free_model(ps, simulator="matrix")
        t0 = time.time()
        mdl.probabilities(c)
        print("densitymx, matrix: %.3fs" % (time.time()-t0))

        #mdl = pygsti.models.create_crosstalk_free_model(ps, simulator="matrix",
        #                                               ideal_gate_type='static unitary')
        #t0 = time.time()
        #mdl.probabilities(c)
        #print("statevec, matrix: %.3fs" % (time.time()-t0))

    if n_qubits <= 12:
        mdl = pygsti.models.create_crosstalk_free_model(ps, simulator="map")
        t0 = time.time()
        mdl.probabilities(c)
        print("densitymx, map: %.3fs" % (time.time()-t0))

    mdl = pygsti.models.create_crosstalk_free_model(ps, simulator="map",
                                                    ideal_gate_type='static unitary')
    t0 = time.time()
    mdl.probabilities(c)
    print("statevec, map: %.3fs" % (time.time()-t0))
    
    mdl = pygsti.models.create_crosstalk_free_model(ps, simulator="map",
                                                    evotype='stabilizer',
                                                    ideal_gate_type='static clifford')
    t0 = time.time()
    out5 = mdl.probabilities(c)
    print("stabilizer, map: %.3fs" % (time.time()-t0))
    
compare_calc_methods(10)

---- Comparing times for 10 qubits (1024 outcomes) ----
Create processor spec: 0.107s
Random Circuit:
Qubit 0 ---|Gx|-|Gx|-|Gy|-|C1|-|Gy|-|C1|-|C1|-|C1|-|Gy|-|Gy|-|Gy|-|Gx|-|C1|-|Gx|-|Gx|-|Gx|-|Gx|-|Gy|-|Gy|-|Gx|---
Qubit 1 ---|C2|-|Gy|-|C2|-|T0|-|Gy|-|T0|-|T0|-|T0|-|C2|-|Gy|-|C2|-|Gy|-|T0|-|Gx|-|Gy|-|Gy|-|Gx|-|C2|-|C2|-|Gy|---
Qubit 2 ---|T1|-|Gy|-|T1|-|Gy|-|Gy|-|Gy|-|C3|-|C3|-|T1|-|Gx|-|T1|-|Gy|-|Gy|-|C3|-|Gy|-|Gy|-|C3|-|T1|-|T1|-|Gy|---
Qubit 3 ---|Gx|-|Gx|-|Gx|-|Gy|-|C4|-|C4|-|T2|-|T2|-|Gy|-|Gx|-|Gx|-|Gy|-|Gy|-|T2|-|Gx|-|C4|-|T2|-|C4|-|C4|-|C4|---
Qubit 4 ---|Gy|-|Gx|-|C5|-|C5|-|T3|-|T3|-|Gx|-|Gy|-|C5|-|Gx|-|Gy|-|Gy|-|Gx|-|Gx|-|Gy|-|T3|-|Gx|-|T3|-|T3|-|T3|---
Qubit 5 ---|C6|-|C6|-|T4|-|T4|-|C6|-|Gy|-|Gy|-|Gy|-|T4|-|Gx|-|Gy|-|Gx|-|C6|-|C6|-|Gy|-|Gx|-|C6|-|Gy|-|Gy|-|Gx|---
Qubit 6 ---|T5|-|T5|-|Gx|-|Gy|-|T5|-|C7|-|C7|-|Gy|-|Gy|-|Gx|-|Gy|-|C7|-|T5|-|T5|-|Gx|-|Gx|-|T5|-|Gx|-|C7|-|Gx|---
Qubit 7 ---|C8|-|C8|-|Gy|-|Gx|-|Gy|-|T6|-|T6|-|Gx|-|C8|-|Gx|-|Gx|-|T6|-|Gy|-|Gy|-|Gy|-|Gy|-|C8|-|C8|

## More qubits
Going beyond 10 qubits, run times will start to get long even for the stabilizer-simulation case.  This is because pyGSTi currently *always* computes *all* the outcomes of a circuit, the number of which scales exponentially with the system size (as $2^n$ for $n$ qubits).  Future versions will remedy this technical issue, allowing you to compute *just* the outcome probabilites you want.  Once this update is released, the stabilizer state simulation will clearly be faster than either the density-matrix or state-vector approaches; for now, we can see that it get's marginally faster as the number of qubits rises.

```{warning}
This cell takes several minutes to run!
```

In [7]:
compare_calc_methods(12)

---- Comparing times for 12 qubits (4096 outcomes) ----
Create processor spec: 0.070s
Random Circuit:
Qubit 0  ---|Gy |-|Gx |-|Gx |-|Gy|-|Gy |-|C1 |-|C1|-|C1|-|C1|-|Gy|-|C1 |-|Gy |-|Gy |-|Gy|-|C1|-|C1|-|Gy |-|Gy |-|C1 |-|Gx |---
Qubit 1  ---|Gy |-|Gx |-|C2 |-|Gx|-|C2 |-|T0 |-|T0|-|T0|-|T0|-|C2|-|T0 |-|C2 |-|Gx |-|Gx|-|T0|-|T0|-|C2 |-|Gy |-|T0 |-|C2 |---
Qubit 2  ---|Gy |-|Gy |-|T1 |-|Gx|-|T1 |-|Gx |-|Gx|-|Gy|-|Gx|-|T1|-|Gy |-|T1 |-|Gx |-|C3|-|Gy|-|Gy|-|T1 |-|Gy |-|Gy |-|T1 |---
Qubit 3  ---|Gx |-|C4 |-|C4 |-|C4|-|Gx |-|C4 |-|Gx|-|C4|-|Gx|-|Gx|-|Gy |-|C4 |-|C4 |-|T2|-|Gy|-|Gy|-|Gy |-|C4 |-|Gy |-|Gx |---
Qubit 4  ---|Gy |-|T3 |-|T3 |-|T3|-|Gx |-|T3 |-|Gx|-|T3|-|Gy|-|Gx|-|Gy |-|T3 |-|T3 |-|Gy|-|C5|-|Gy|-|C5 |-|T3 |-|C5 |-|Gx |---
Qubit 5  ---|C6 |-|C6 |-|Gx |-|C6|-|Gy |-|C6 |-|C6|-|Gx|-|Gy|-|Gy|-|Gy |-|Gx |-|Gy |-|Gx|-|T4|-|C6|-|T4 |-|Gx |-|T4 |-|C6 |---
Qubit 6  ---|T5 |-|T5 |-|C7 |-|T5|-|Gx |-|T5 |-|T5|-|Gy|-|C7|-|Gx|-|Gx |-|Gy |-|C7 |-|C7|-|Gx|-|T5|-|Gx |-|Gy |-|Gx |-|T5 |---
Qubit 7  

In [8]:
#compare_calc_methods(16)